# Figure 2 — What is happening during the run, in triplicate
Journals (and good practice) want **independent repeats**. We run three trajectories from the same minimized structure with different initial velocities (seeds 2024/2025/2026), look at the first in detail, then overlay all three to see the spread you should expect: the *ensemble* behaviour is consistent even though individual events (a hydrogen bond breaking, a dihedral wandering) happen at different times in different runs. That is what MD reproducibility looks like — statistical, not frame-by-frame.

**Reproducibility note (subtler than it looks).** On CUDA with `DeterministicForces`, dynamics *and* minimization reproduce **bit-for-bit from a fixed start** on the same GPU model. The non-obvious trap is **system preparation**: `addHydrogens` places hydrogens at random positions then relaxes them on a fast non-deterministic platform, and `addSolvent` places ions with `random.choice` — so a fresh prep drifts run-to-run and MD chaos amplifies it into divergent trajectories. The fix (baked into `mdtutorial`): seed `random`+`numpy` *and* pin the hydrogen fix-up to the deterministic **Reference** platform. Then the whole from-scratch pipeline reproduces from the seed — but only **per GPU model** (floating-point non-associativity means bitwise identity never survives a change of hardware; across machines you rely on **statistical** reproducibility, the scientific standard the three repeats show). `md_determinism_test.ipynb` measures all of this directly.

**Two ways to run this notebook (the intro's two tiers).** By default (*live tier*) it loads the system Figure 1 saved and simulates short trajectories — so **run Figure 1 first**, and it errors clearly if you haven't. Set **`LOAD_REFERENCE = True`** to instead analyze the shipped multi-ns **reference** trajectories (`REF_ROOT`): no GPU and no Figure 1 needed — the same figures below, computed on runs long enough to have honest statistics.

In [ ]:
# --- environment on-ramp: make sure the MD stack + the module are importable in THIS kernel ---
import importlib.util, sys, os, subprocess
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:                          # Colab: provision the stack
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:                                                            # still missing -> almost always the WRONG KERNEL
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/todd471/MD_tutorial/main")
for _mod in ("mdtutorial.py", "mdtviz.py", "md_scalogram.py"):          # grab the shipped modules if absent
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import mdtutorial as mdt, mdtviz
PYMOL = mdtviz.setup_pymol()                                            # find/provision headless PyMOL (panels skip if none)

In [ ]:
# --- configuration ---
import numpy as np, mdtraj as md
import matplotlib.pyplot as plt
OUT  = "trpcage_out"   # scratch root: prep, figures, trajectories. Local -- on Colab it's the VM's ephemeral disk.
PREP = OUT             # where fig1 SAVES its prepared system and fig2/fig3 LOAD it. Locally the notebooks share one
                       # filesystem so the handoff is free. On Colab each notebook is a separate VM, so fig2/fig3
                       # just re-prepare their own system when fig1's isn't present (see the load cell) -- no Drive
                       # needed. To reuse fig1's EXACT prep across notebooks on Colab, point PREP at a Google Drive
                       # path instead (opt-in; see the README's Colab section).
N_PROD_PS = int(os.environ.get("N_PROD_PS", "200"))   # production length per repeat (ps); override for a quick test
SEEDS     = [2024, 2025, 2026] # independent repeats -- EDIT this list to change which/how many (the run count
                               #   is just len(SEEDS)). For the 6-seed canonical reference use the six-seed list:
                               #   [2021, 2022, 2023, 2024, 2025, 2026].
RUN_MODE  = "interactive"      # "canonical" ALSO writes the archival slate (checkpoints/system.xml/integrator/
                               #   final_state/run_meta) to OUT -- set it with the six-seed SEEDS above +
                               #   N_PROD_PS=10000 to GENERATE the reference. Writes to OUT only; never touches REF_ROOT.
REF_ROOT  = "reference_10ns"   # SHIPPED read-only reference (6 seeds x 10 ns); §2.6/timescales load it for an honest
                               #   autocorrelation time. Separate from OUT, so a canonical rerun never overwrites it.
LOAD_REFERENCE = False         # False = live tier (load fig1's prep + simulate). True = analyze the REF_ROOT
                               #   trajectories instead (no GPU, no fig1): the same figures on multi-ns runs.
CANON_SEED = 2024              # example seed for the §2.6 timescale + §2.6b scalogram (2024 caught the notable
                               #   Ser13 rotamer flip -- a rare event only one seed sampled).
PLOT_SEEDS = [2024, 2025, 2026]  # which seeds to DRAW in the §2.5 overlay -- ALL loaded seeds are still analyzed;
                               #   this only declutters the plot. Edit to show any subset (the other seeds are there).

### Get the trajectories — live tier (simulate) or reference tier (load)
Default (*live tier*): load Figure 1's prepared system and simulate. Figure 1 owns preparation; this notebook only **loads** what it saved (`load_prepared` errors clearly if you haven't run Figure 1 — it will not silently re-prepare, and the solvation seed lives in Figure 1's `SEED`, not here). With **`LOAD_REFERENCE = True`** (*reference tier*): load the shipped multi-ns trajectories from `REF_ROOT` instead — no prep, no GPU. Either way, everything below just reads `cvs`.

In [ ]:
if LOAD_REFERENCE:
    cvs_all, RUN_SEEDS, DT = mdt.load_reference(REF_ROOT)   # analyze the shipped long run: no GPU, no Figure 1
    prep = None
    print(f"reference tier: {len(cvs_all)} seeds x {DT * cvs_all[0]['t'].n_frames / 1000:g} ns from {REF_ROOT}/")
else:
    prep = mdt.load_or_prepare(PREP, OUT)                  # reuse fig1's prep if present, else prep a fresh one here (self-sufficient)
    cvs_all, RUN_SEEDS, DT = None, SEEDS, 1.0

### 2.1  The first trajectory
*Live tier:* a fresh Context (so its random stream is reproducible) writes a trajectory + a scalar state log, and we compute the collective variables from it. *Reference tier:* we take the first shipped trajectory. Either way we look at this one in detail before overlaying the rest.

*Code: `run_repeat` (in `mdtutorial.py`) runs one trajectory in a **fresh** OpenMM Context with a `LangevinMiddleIntegrator` (the LFMiddle scheme; Zhang *et al.* 2019) — a fresh Context per run is exactly what makes the seed reproducible; `compute_cvs` derives the observables with **MDTraj**.*

In [ ]:
if LOAD_REFERENCE:
    cvs = [cvs_all[0]]
else:
    cvs = [mdt.compute_cvs(mdt.run_repeat(prep, RUN_SEEDS[0], n_prod_ps=N_PROD_PS, run_mode=RUN_MODE, out_root=OUT),
                           top=mdt.outp("stage4_minimized.pdb", PREP))]
print(f"seed {RUN_SEEDS[0]}: RMSD end {cvs[0]['rmsd'][-1]:.1f} A, helix {cvs[0]['helix'].mean():.2f}")

### 2.1c  What a run writes to disk
A run leaves **files** on disk, not just a plot (whether you simulated it here or loaded the shipped reference). MD writes **two kinds of data**, and every figure later in this notebook pulls from one of them:

| output | what it is | you read it for |
|---|---|---|
| **trajectory** — `traj_<seed>.dcd` | atomic **coordinates** (every ps in a live run) | structure & motion: RMSD, Rg, H-bonds, the animation — all of §2.2 onward |
| **state log** — `state_<seed>.csv` | scalar **thermodynamics** every ps: potential/kinetic/total energy, temperature, box volume, density | the run's *vital signs* — did it equilibrate, is the thermostat holding, is the box sane |
| *(canonical mode only)* `checkpoint_*.chk`, `system.xml`, `integrator_<seed>.xml`, `final_state_*.xml` | the exact **inputs + restart state** | extending or reproducing the run bit-for-bit (see `md_determinism_test.ipynb`) |

The trajectory gets all the attention, but the **state log is your sanity check** — after a run (or *live*, by tailing the file on a long cluster job) it tells you the coordinates you are about to analyze came from a physically sane, equilibrated system rather than one slowly cooking or exploding. Our runner records production *after* a short NVT equilibration. A healthy log is **not** a flat line — it is **noisy but stationary**: the *mean* holds steady while the instantaneous values scatter around it. That scatter is real thermal fluctuation, and it is large here because the system is small (a few thousand atoms) — the temperature swings ±4 K around 300 K, which is *precisely* the canonical fluctuation σ_T = T·√(2⁄N_dof) for a system this size (Lebowitz *et al.* 1967; Frenkel & Smit), and the potential energy ±0.4%. Stability is the flat *mean* (the run below drifts < 0.05% end-to-end), not a flat trace.

In [ ]:
_md_dir = os.path.join(REF_ROOT if LOAD_REFERENCE else OUT, "md_output")   # live run, or the reference bundle
print("files in", _md_dir + "/ :")
for _f in sorted(os.listdir(_md_dir)):                     # what MD actually produced (trajectory + state log)
    print(f"    {_f:26s} {os.path.getsize(os.path.join(_md_dir, _f)) / 1e6:8.2f} MB")
D = np.genfromtxt(os.path.join(_md_dir, f"state_{RUN_SEEDS[0]}.csv"), delimiter=",", skip_header=1)   # step,time,PE,KE,E,T,vol,rho,speed
t, pe, te, temp, vol = D[:, 1] / 1000.0, D[:, 2], D[:, 4], D[:, 5], D[:, 6]
_drift = np.polyfit(t, pe, 1)[0] * (t[-1] - t[0]) / abs(pe.mean()) * 100   # % change of the PE mean over the whole run
fig, ax = plt.subplots(2, 2, figsize=(10, 6.2), sharex=True, constrained_layout=True)
panels = [(ax[0, 0], pe,   "C0", "kJ/mol",  f"potential energy — mean {pe.mean():.0f} (±{pe.std()/abs(pe.mean())*100:.1f}% thermal)"),
          (ax[0, 1], temp, "C3", "K",       f"temperature — {temp.mean():.1f} ± {temp.std():.1f} K"),
          (ax[1, 0], te,   "C2", "kJ/mol",  "total energy — NVT (exchanged w/ thermostat, not conserved)"),
          (ax[1, 1], vol,  "C4", "nm$^3$",  "box volume — flat (NVT, no barostat)")]
for a, y, col, yl, ttl in panels:
    a.plot(t, y, lw=0.5, color=col)
    a.axhline(y.mean(), color="k", ls="--", lw=1.2)        # the MEAN: flat mean = stable; the scatter around it is thermal
    a.set_title(ttl, fontsize=9.5); a.set_ylabel(yl)
ax[0, 1].set_ylim(temp.mean() - 20, temp.mean() + 20)      # ±20 K context so the ±4 K thermal swing reads as small
ax[1, 1].set_ylim(vol.mean() - 0.06, vol.mean() + 0.06)    # tight window: a constant NVT volume plots as the flat line it is
ax[1, 0].set_xlabel("time (ns)"); ax[1, 1].set_xlabel("time (ns)")
fig.suptitle("Run vital signs — the MEAN holds; the scatter is expected thermal noise (small system)", fontsize=12)
fig.savefig(mdt.outp("figure_state_log.png", OUT), dpi=130); plt.show()
print(f"health: PE {pe.mean():.0f} kJ/mol (±{pe.std()/abs(pe.mean())*100:.1f}%, drift {_drift:+.2f}% over run) | "
      f"T {temp.mean():.1f}±{temp.std():.1f} K (canonical σ ≈ T√(2/Ndof)) | V spread {vol.std():.4f} nm³ (NVT: flat)")

### 2.2  Watch it happen — molecule and observables, synchronized
One play button, one timeline; the grey vertical line marks the current moment on every plot. Backbone coloured N→C; the two dashed lines are backbone α-helix hydrogen bonds — **GLN5 N–H ··· ASN1 O=C** (orange, matching its trace) and **ASP9 N–H ··· GLN5 O=C** (blue) — donor N–H of residue *i* to the carbonyl O of residue *i*−4. Each line is **bold-dashed when the bond is satisfied and faint-dotted when it breaks**, where "satisfied" is the **H···O distance ≤ 2.5 Å** — a distance-only proxy (a strict H-bond criterion would add a donor–H···acceptor angle term, but distance alone is plenty to *watch* the bond come and go). Watch GLN5's orange line go dotted exactly as its orange curve spikes past the cutoff.

*Code: `player` (in `mdtviz.py`) builds the synchronized figure with **Matplotlib**'s `FuncAnimation` and embeds it as an HTML5 video; long runs are subsampled to ≤ 200 video frames while the traces stay full-resolution.*

In [ ]:
mdtviz.player(cvs[0])

### 2.3  Ray-traced filmstrip (print output, repeat 1)
Four aligned snapshots along the run, PRO19 highlighted in magenta (skips gracefully if PyMOL isn't available).

*Code: `filmstrip` (in `mdtviz.py`) ray-traces cartoon snapshots with headless open-source **PyMOL**, all superposed to frame 0; skips cleanly if PyMOL isn't installed.*

In [ ]:
import matplotlib.image as mpimg
pngs, frames = mdtviz.filmstrip(cvs[0]["t"], os.path.join(OUT, "figures"))
if pngs:
    figF, axF = plt.subplots(1, len(pngs), figsize=(4 * len(pngs), 4.3), facecolor="white")
    for axf, f, fr in zip(np.atleast_1d(axF), pngs, frames):
        axf.imshow(mdtviz.sqcrop(mpimg.imread(f))); axf.axis("off"); axf.set_title(f"t = {cvs[0]['ps'][fr]} ps", fontsize=12)
    figF.suptitle("Repeat 1 filmstrip (PRO19 in magenta)", fontsize=13); figF.tight_layout()
    figF.savefig(mdt.outp("figure2_filmstrip.png", OUT), dpi=150, bbox_inches="tight"); plt.show()

### 2.4  Now the remaining independent repeats
Same start, different initial velocities — the repeats the journal asks for (live tier simulates them; reference tier just loads the rest of the shipped seeds).

In [ ]:
for k, s in enumerate(RUN_SEEDS[1:], start=1):
    if LOAD_REFERENCE:
        cvs.append(cvs_all[k])
    else:
        dcd = mdt.run_repeat(prep, s, n_prod_ps=N_PROD_PS, run_mode=RUN_MODE, out_root=OUT)
        cvs.append(mdt.compute_cvs(dcd, top=mdt.outp("stage4_minimized.pdb", PREP)))
    print(f"seed {s}: RMSD end {cvs[-1]['rmsd'][-1]:.1f} A, helix {cvs[-1]['helix'].mean():.2f}")

### 2.5  All repeats together — the spread you should expect
Six observables, three global and three local. **Top row** (global fold): **Cα RMSD** drifts to a similar plateau, radius of gyration and DSSP helix fraction hold steady — the fold is stable in every run. **Bottom row** (local, deliberately contrasting timescales): the **PRO19 ψ** dihedral mostly sits still and only *occasionally* makes a big discrete flip — you may see one, or none, in any given short run, and a flip need not stay flipped; **SER13 ψ** (in the marginal 3₁₀-helix) jitters *constantly* but with small amplitude; and the canonical **ASP9–ARG16 salt bridge** makes big, discrete excursions — the ARG16 side chain swings in and out on its long, solvent-exposed arm, so the distance jumps from about 3 Å (formed) to as much as nearly 1 nm (fully broken). The jumps are large but their *rate* varies wildly from seed to seed (across our reference runs, broken anywhere from roughly 2% to 30% of the time) — which makes it the clearest reminder in the set to **never trust a single trajectory**. The shared story: the *ensemble* behaviour is consistent across seeds, but the exact timing of every excursion differs — and each observable lives on a different timescale, which is exactly what the §2.6 timescale analysis makes precise. (Dihedrals are re-centered off the ±180° seam so thermal wiggles across it don't masquerade as huge flips.)

*Code: every observable comes from `compute_cvs` (in `mdtutorial.py`) — Cα RMSD, H-bond and salt-bridge distances, and the backbone dihedrals via **MDTraj**, helix fraction via simplified **DSSP**; Rg is MDTraj's `compute_rg`, and dihedrals are unwrapped off the ±180° seam with `circmean_deg` / `recenter_deg`.*

In [ ]:
rc = lambda a: mdt.recenter_deg(a, mdt.circmean_deg(a))
_PAL = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#17becf"]   # 6 distinct colors, no muddy brown
fig, ax = plt.subplots(2, 3, figsize=(14, 6.6), sharex=True)
for k, c in enumerate(cvs):
    if RUN_SEEDS[k] not in PLOT_SEEDS:                     # PLOT_SEEDS declutters the overlay; every seed is still analyzed
        continue
    st = max(1, len(c["ps"]) // 2000)                     # thin dense traces so the overlay stays legible
    p = c["ps"][::st]; kw = dict(lw=0.6, alpha=0.75, color=_PAL[k % len(_PAL)], label=f"seed {RUN_SEEDS[k]}")
    ax[0, 0].plot(p, c["rmsd"][::st], **kw)
    ax[0, 1].plot(p, (md.compute_rg(c["t"]) * 10)[::st], **kw)
    ax[0, 2].plot(p, c["helix"][::st], **kw)
    ax[1, 0].plot(p, rc(c["p19"])[::st], **kw)
    ax[1, 1].plot(p, rc(c["ser13"])[::st], **kw)
    ax[1, 2].plot(p, c["d9r16"][::st], **kw)
ax[0, 0].set(title="Cα RMSD", ylabel="Å"); ax[0, 1].set(title="radius of gyration", ylabel="Å")
ax[0, 2].set(title="helix fraction (DSSP)", ylabel="fraction")
ax[1, 0].set(title="PRO19 ψ — rare slow flip", xlabel="time (ps)", ylabel="deg")
ax[1, 1].set(title="SER13 ψ (3₁₀) — fast jitter", xlabel="time (ps)", ylabel="deg")
ax[1, 2].set(title="ASP9–ARG16 salt bridge (slow)", xlabel="time (ps)", ylabel="Å")
_h, _l = ax[0, 0].get_legend_handles_labels()                     # one shared legend ABOVE the grid, off the data
fig.legend(_h, _l, loc="upper center", ncol=len(cvs), fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(mdt.outp("figure2_dynamics.png", OUT), dpi=130, bbox_inches="tight"); plt.show()

### 2.6  Can we trust these averages? — convergence & uncertainty
The overlaid traces *look* consistent, but "looks consistent" is not a measurement. The number that makes it one is the **standard error of the mean (SEM)** — how far the average you just computed is likely to sit from the true average. The textbook SEM is σ/√N, but that assumes your N frames are N *independent* draws, and they are not: consecutive frames are highly correlated. Two questions fix it:

- **How many *independent* samples do you actually have?** The **integrated autocorrelation time** τ counts how long the observable takes to forget itself. Its autocorrelation ρ(k) starts at 1 and decays as frames decorrelate; **τ = ½ + Σₖ ρ(k)** sums that decay (the ½ is the zero-lag self-term). In an infinite run ρ(k) → 0, but a *finite* run's ρ(k) turns to noise at large lag and wanders below zero — summing that noise only corrupts τ, so we stop at the first crossing (an *initial-positive-sequence* cutoff; Geyer 1992, Sokal 1997). The effective sample count is **N_eff = N ⁄ 2τ** (white noise → τ = ½ → N_eff = N).
- **What is the honest error bar?** **SEM = σ ⁄ √N_eff = σ·√(2τ ⁄ N)** — the naïve σ/√N is too small by the factor √(2τ).

**Estimate τ on a run long enough to see the slow motions.** τ and N are the same kind of thing — both counts of frames: τ is how many frames it takes to forget, N is how many you ran. So the honest comparison is a *ratio*: a clean answer needs the run to be *many* correlation times long (N ⁄ τ large). The trap is that Rg's autocorrelation is a *sum* of relaxations — side chains (picoseconds), loops (nanoseconds), substate hops (ns–µs) — so a short window sees only the fast ones and τ comes out far too small. The cell below prints it for this run: on a 200 ps slice τ is tiny, on the full trajectory it is far larger — a short run *hides its own correlation time*. That is why we quote convergence from the multi-ns **reference** (`REF_ROOT`), not the live run above.

**Why τ — and so the SEM — is a *lower bound*.** Two effects push the same way. (1) A finite run cannot see relaxations slower than itself, so slow modes beyond your run length are simply missing from τ; run longer and τ grows. (2) The estimator stops at the *first* noisy zero-crossing, which can cut a real slow tail short before it has fully decayed. Both make the measured τ an underestimate — so the SEM you get is the *smallest* the true uncertainty could be, a floor and never a ceiling.

**The block-averaging picture — and why it usually can't plateau.** Both panels below plot a **standard error against block size** — the *error bar on the average*, not the observable levelling off. Block averaging: chop the run into blocks of size *b*, average each block, and take SEM = std(block means)/√(number of blocks). Once each block is longer than the memory (*b* > τ), non-overlapping blocks are effectively independent, so the SEM climbs to a **plateau** — and *that plateau is the true error* (left, a **well-sampled reference case**: a toy series with a short, *known* τ and plenty of samples, so it does plateau — what success looks like). But our real run (right, Rg) is only a few correlation times long, so the curve never cleanly plateaus, and the **Flyvbjerg–Petersen error bars** on its tail — ballooning as the blocks run out — show it simply *cannot* be pinned down that way. With no plateau, the τ-based SEM (the red line) is the number to quote: it needs only the one trajectory. The lower grey line is the naïve σ/√N — the SEM you would get if the frames were *independent*, i.e. if you **shuffled** the trajectory and destroyed its time order. The real curve climbs above that floor by exactly √(2τ), and *that climb is how far from uncorrelated your run is* — a measure of correlation, not of whether you covered the right conformations (that is the seed cross-check next).

Then the check a single trajectory can never do for itself: **do independent seeds agree within these error bars?** The cross-check below spans the reference's independent seeds — if they land on the same average within their τ-based SEMs, that is real evidence of convergence; if the seed-to-seed spread exceeds the bars, the run is undersampled no matter how tight any single bar looks.

**Where the code and the math live.** τ is `integrated_autocorr_time`, the SEM is `trust_report` (both in `mdtutorial.py`); the SEM-vs-block-size curve *with* its Flyvbjerg–Petersen error bars ±SEM/√(2(M−1)) is `block_curve`, and the two-panel figure that calls it is `mdtviz.convergence_figure`. τ uses Sokal's τ = ½ + Σρ definition with Geyer's initial-positive-sequence truncation (sum consecutive pairs, stop at the first non-positive pair) — that truncation is what makes τ conservative. Conventions and best practice: **Sokal 1997**, **Geyer 1992** (τ), **Flyvbjerg & Petersen 1989** (block averaging), **Grossfield *et al.* 2018** (uncertainty), **Chodera 2016** (equilibration detection).

In [ ]:
# Convergence is quoted from the LONG reference, never the short live run: a 200 ps window sees
# only fast motions, so its Rg autocorrelation time comes out ~50x too short (see prose). On the reference
# tier cvs IS the reference; on the live tier we load it anyway (the live tau would be untrustworthy).
if LOAD_REFERENCE:
    ref, ref_seeds = cvs, RUN_SEEDS                        # already the reference (DT set in "Get the trajectories")
    src = f"{len(ref)}-seed reference ({DT:g} ps/frame)"
else:
    try:
        ref, ref_seeds, DT = mdt.load_reference(REF_ROOT)
        src = f"{len(ref)}-seed {DT * ref[0]['t'].n_frames / 1000:g} ns reference ({DT:g} ps/frame)"
    except FileNotFoundError as e:
        ref, ref_seeds, DT = cvs, RUN_SEEDS, 1.0
        src = f"{N_PROD_PS} ps LIVE run -- NO reference bundle found, so τ is biased short!"
        print("!!", e)
print("convergence source:", src)

# Why the reference, not the 200 ps live run: τ balloons as you look at more of the trajectory (see prose).
_canon = next((c for c in ref if c["seed"] == CANON_SEED), ref[0])       # the canonical example seed (config)
_rg = md.compute_rg(_canon["t"]) * 10
_ts = mdt.integrated_autocorr_time(_rg[:max(8, int(200 / DT))]) * DT     # τ from a 200 ps slice
_tf = mdt.integrated_autocorr_time(_rg) * DT                             # τ from the full run
print(f"τ(Rg) = {_ts:.0f} ps on a 200 ps slice  vs  {_tf:.0f} ps on the full {DT * len(_rg) / 1000:g} ns "
      f"({_tf / _ts:.0f}× — a short run hides its own correlation time)")

# The two-panel block-averaging figure — idealized plateau (left) vs. the real block curve with
# Flyvbjerg–Petersen error bars (right), plus the τ-based SEM floor — lives in mdtviz.convergence_figure.
# The estimator math (τ, N_eff, block SEM, the FP error bars) is in mdtutorial; see the §2.6 prose.
mdtviz.convergence_figure(md.compute_rg(_canon["t"]) * 10, dt_ps=DT, label="Reference Rg",
                          out_png=mdt.outp("figure_convergence.png", OUT)); plt.show()

# The one check a single run can't do for itself: do independent seeds agree within their τ-based bars?
print(f"\nRg per seed (mean ± τ-based SEM) — do independent seeds agree within their error bars?")
_stats = []
for c in ref:
    tag = f"  seed {c['seed']}" + (f" / {c['solvation']}" if "solvation" in c else "")
    _stats.append(mdt.trust_report(md.compute_rg(c["t"]) * 10, dt_ps=DT, label=tag))
_spread = np.ptp([s["mean"] for s in _stats]); _sem = np.mean([s["sem"] for s in _stats])
print(f"\nverdict: seed-to-seed spread {_spread:.3f} Å vs a typical error bar {_sem:.3f} Å  ->  " + (
      "CONVERGED (the spread fits inside the bars)." if _spread < 2 * _sem else
      "UNDERSAMPLED (the spread is bigger than the bars, so even this run isn't long enough for Rg to this precision)."))

### 2.6b  Where do those timescales live? — one observable, many scales
The τ in §2.6 collapses an observable's *entire* spectrum of relaxations into a single number. This section spreads it back out — resolved in **timescale** and localized in **time** — so you can see whether that one number describes a single clean motion or hides several. We switch observable to the side-chain **χ1 dihedrals** (one per residue): they are the local clocks of the fold: a single χ1 jitters within its rotamer well on picosecond timescales, hops between wells far more rarely, and inherits slower timescales still from the collective motions it couples to — a hierarchy of motions across spatial scales, not one clock.

**The scalogram.** A wavelet transform partitions a series' fluctuation across a (timescale × time) grid. Steady fast jitter puts power at short timescales across the whole width; a rare rotamer hop puts a **burst at a long timescale, localized at the instant it happens**. Two flavors (both in `md_scalogram.py`): the **blocking / Haar DWT** (orthonormal — an *exact* variance partition, but dyadic and blocky) and the **Morlet CWT** (continuous and smooth, at the cost of being only *near*-variance). The strip below uses the CWT for the smoother picture. Either flavor answers the *variance-partition* question — how an observable's variance splits across timescales (the Percival & Walden **wavelet variance**) — which is **not** §2.6's error-bar-on-the-mean: that used the *averaging* branch of the very same Haar transform (**Flyvbjerg–Petersen** block averaging), this uses the *differencing* branch. Same machinery, different questions — and the √2 Haar normalization only sets the bookkeeping (Parseval: the partition sums to the total variance), it changes no interpretation.

**The marginal.** Project the scalogram onto the timescale axis — the mean power at each scale — and you get the **global wavelet spectrum**, drawn flush on the right of each panel: the red curve is the smooth CWT spectrum — literally the color beside it summed onto scale — and the grey bars behind it are the **exact** Haar-DWT variance per octave: the quantitative partition to the CWT's qualitative guide (the `HAAR` knob toggles overlay / off / only). It answers the question the single τ provokes — *does this observable's variance sit at one timescale or spread across many?*

**Reading the strip (left → right = fast → slow).** Residues are ranked by their χ1 τ and sampled at the five **quartile boundaries** of that ranking — fastest, Q1, median, Q3, slowest (five points bracketing four quartiles, not quintiles). Watch the marginal's mass **climb off the floor toward longer timescales** as τ grows, with the dashed **2τ** line (the §2.6 number) riding up alongside it. On the left, a fast residue keeps its power low and broad with a large **N_eff** — trust that τ. On the right, the mass migrates up into a narrow **rare-event lobe** carried by a handful of hops and N_eff collapses to a few: the single τ there is *flagging a rare event*, not measuring a persistent slow mode — the per-residue version of §2.6's UNDERSAMPLED verdict.

**The convergence reading.** For a converged observable you *want* most of the power down at the low timescales; a **slow-timescale-heavy marginal is a red flag** that the run is being dominated by rare events it hasn't sampled enough of. Most residues here are honest and fast; the slow tail is exactly where a longer run is still owed.

**The honest fine print.** (1) The faded **cone of influence** in each panel's top corners is where the wavelet overran the ends of the record and began reading padding rather than signal — do not trust power up there (Torrence & Compo 1998). (2) The CWT marginal is a **near-variance, qualitative** spectrum, *not* a normalized partition: the redundant transform only ≈recovers the variance through the reconstruction factor C_δ = 0.776, and the [2, N⁄4]-frame scale window plus the cone discount it further — for an *exact* variance-by-scale accounting use the blocking/Haar marginal (Parseval). (3) χ1 is an **angle**: its power is built from cos/sin, never from `np.unwrap` (which would manufacture slow power out of the ±180° seam). (4) None of this resolves on a 200 ps run — the strip needs the multi-ns **reference**, and is skipped with a note if it isn't present.

**Where the code and the math live.** The fast→slow residue selection is `md_scalogram.chi1_quartiles`; the strip is `md_scalogram.marginal_strip_figure`; underneath, `cwt_scalogram` / `blocking_scalogram` build the two transforms, `morlet_coi` draws the cone, and `cwt_marginal` computes the cone-respecting spectrum. Wavelet power spectrum, cone of influence and C_δ: **Torrence & Compo 1998**; the 1⁄s scale-bias correction: **Liu *et al.* 2007**; blocking = Haar wavelet variance: **Flyvbjerg & Petersen 1989** with **Percival & Walden 2000**.

In [ ]:
# 2.6b needs a genuinely long trajectory to resolve slow χ1 motions; reuse the reference
# loaded in §2.6 (ref/DT). If only a short live run is present, SKIP with a note (like the PyMOL-absent
# case) rather than draw an uninterpretable 200 ps strip that a reader could mistake for signal.
import md_scalogram as msc
_ex = next((c for c in ref if c["seed"] == CANON_SEED), ref[0])         # CANON_SEED = the chosen example (config)
if _ex["t"].n_frames * DT < 2000:                                       # need >= ~2 ns to see the slow lobe
    print("§2.6b skipped: it needs the long (multi-ns) reference to resolve multi-scale χ1 dynamics.")
    print(f"  The loaded trajectory is only {_ex['t'].n_frames * DT:g} ps. Set LOAD_REFERENCE=True, or make the")
    print(f"  shipped {REF_ROOT}/ reference bundle available (see README), to render this figure.")
else:
    # Rank χ1 by circular autocorrelation time, take 5 evenly spaced fast->slow, strip their scalograms +
    # marginals. All the transform / COI / marginal math lives in md_scalogram (see prose).
    _series, _labels, _taus = msc.chi1_quartiles(_ex["t"], nq=5)
    HAAR = "overlay"      # marginal lever: "overlay"=CWT + exact Haar bars (quantitative) | "off"=smooth CWT only | "only"=exact Haar-DWT strip
    _fig = msc.marginal_strip_figure(_series, _labels, dt=DT, circular=True, obs_label="χ1 (deg)",
        method=("blocking" if HAAR == "only" else "cwt"), blocking_marginal=(HAAR == "overlay"),
        title=f"seed {_ex['seed']}: χ1 scalograms + marginals across τ quartile boundaries (fast → slow)")
    _fig.savefig(mdt.outp("figure_timescales.png", OUT), dpi=110, bbox_inches="tight"); plt.show()
    print("quartile boundaries (fast→slow): " + "  ".join(f"{l} τ≈{tau*DT:.0f} ps" for l, tau in zip(_labels, _taus)))